# Stiffness-label feasibility audit
Recorded evidence review. Run from the workspace root or audit_stiffness. See ASSESSMENT.md for interpretation and source limitations.
The extraction scripts read originals and write only audit outputs. This notebook verifies the cached audit, not a new raw-data extraction.

In [1]:
from pathlib import Path
import json
import pandas as pd
audit = Path.cwd() if (Path.cwd() / 'specimen_summary.csv').exists() else Path.cwd() / 'audit_stiffness'
summary = pd.read_csv(audit / 'specimen_summary.csv')
labels = pd.read_csv(audit / 'label_checkpoints.csv')
channels = pd.read_csv(audit / 'strain_channels.csv')
metadata = json.loads((audit / 'pzt_metadata.json').read_text())
assert summary.pzt_files.sum() == 1495
assert summary.strain_mat.sum() == 317
assert len(labels) == 39
assert not labels.duplicated(['specimen', 'cycles']).any()
assert labels.distinct_values.eq(1).all()
assert labels.referenced_strain_files.eq(labels.resolved_strain_files).all()
assert channels.finite.eq(channels.n).all()
assert all(r['path_count'] == 252 for r in metadata)
print(summary.groupby('layup')[['pzt_files','scalar_checkpoints','positive_checkpoints']].sum().to_string())


        pzt_files  scalar_checkpoints  positive_checkpoints
layup                                                      
Layup1        516                  29                    27
Layup2        453                  10                     8
Layup3        526                   0                     0


In [2]:
print(labels[['specimen','cycles','stiffness_degradation','status']].to_string(index=False))
assert labels.stiffness_degradation.gt(0).sum() == 35
assert labels.stiffness_degradation.eq(0).sum() == 3
assert labels.stiffness_degradation.lt(0).sum() == 1
assert summary.scalar_label_files.sum() == 255
assert summary.zero_vector_files.sum() == 49
assert summary.empty_label_files.sum() == 1191
print('39 checkpoint candidates = 35 positive + 3 baseline references + 1 negative flag.')
print('These counts do not establish independent mechanical validation.')


specimen  cycles  stiffness_degradation                     status
L1_S11_F       0               0.000000    baseline_reference_only
L1_S11_F       1               0.055036 published_scalar_candidate
L1_S11_F      10               0.045813 published_scalar_candidate
L1_S11_F     100               0.060736 published_scalar_candidate
L1_S11_F    1000               0.069766 published_scalar_candidate
L1_S11_F   10000               0.076006 published_scalar_candidate
L1_S11_F   20000               0.098380 published_scalar_candidate
L1_S11_F   30000               0.100517 published_scalar_candidate
L1_S11_F   40000               0.112262 published_scalar_candidate
L1_S11_F   50000               0.118554 published_scalar_candidate
L1_S11_F   60000               0.104267 published_scalar_candidate
L1_S11_F   70000               0.128221 published_scalar_candidate
L1_S11_F   80000               0.122361 published_scalar_candidate
L1_S11_F   90000               0.114456 published_scalar_candi

In [3]:
# Independently transcribed from the supplied Technical Report, Appendix A.
reference = {
 'L1_S11_F': ([0,1,10,100,1000,10000,20000,30000,40000,50000,60000,70000,80000,90000,100000],
 [0,.055036381,.045813245,.060735707,.069766198,.076005931,.098380137,.10051699,.11226211,.118553783,.10426744,.128221136,.12236086,.11445585,.120155531]),
 'L1_S12_F': ([0,1,10,100,1000,10000,30000,40000,50000,60000,70000,80000,90000,100000],
 [0,.043288345,.051679586,.047985147,.059726484,.084112497,.233175942,.239071597,.236134709,.245338988,.244799668,.237792304,.245462847,.245730332]),
 'L2_S11_F': ([0,1,10,100,1000,10000,20000,30000,40000,50000,60000],
 [0,-.012213572,.012154654,.005044283,.008583815,.010634185,.031656763,.042811126,.055037786,.044292129,.084180661])}
for specimen, (cycles, values) in reference.items():
    expected = dict(zip(cycles, values))
    actual = labels[labels.specimen.eq(specimen)]
    assert all(abs(row.stiffness_degradation-expected[row.cycles]) < 1e-12 for row in actual.itertuples())
    print(specimen, 'matching checkpoints:',len(actual), 'documented cycles missing in actual labels:', sorted(set(cycles)-set(actual.cycles)))


L1_S11_F matching checkpoints: 15 documented cycles missing in actual labels: []
L1_S12_F matching checkpoints: 14 documented cycles missing in actual labels: []
L2_S11_F matching checkpoints: 10 documented cycles missing in actual labels: [100]


In [4]:
conflicts = pd.read_csv(audit / 'metadata_conflicts.csv')
print(conflicts.groupby(['specimen','kind']).size().to_string())
bad = channels[(channels.specimen.eq('L3_S11_F')) & (channels.file.eq('L3_S11_F08_DAT.mat')) & channels.channel.eq('strain1')]
assert len(bad)==1 and bad.iloc[0]['min']==bad.iloc[0]['max'] and bad.iloc[0]['n']==92550
print('Confirmed constant L3S11 F08 channel 1:', bad[['n','min','max']].to_string(index=False))
integrity = json.loads((audit / 'source_integrity.json').read_text())
assert integrity['changed_size_or_mtime'] == []
print('Source files with unchanged size and last-write timestamp:',integrity['files_checked'])


specimen  kind 
L1_S18_F  load      1
L2_S18_F  cycle    89
          load     83
L3_S11_F  load      1
Confirmed constant L3S11 F08 channel 1:     n      min      max
92550 0.039683 0.039683
Source files with unchanged size and last-write timestamp: 2571
